# vLLM + AITER + ROCm on Strix Halo

Welcome! This image (`ghcr.io/amdresearch/auplc-vllm:latest`) is a from-source build of:

- **[vLLM](https://github.com/vllm-project/vllm)** — high-throughput LLM serving engine with paged KV-cache and an OpenAI-compatible API.
- **[ROCm flash-attention](https://github.com/ROCm/flash-attention)** (`main_perf` branch) — the Triton AMD attention backend.
- **[AITER](https://github.com/ROCm/aiter)** — AI Tensor Engine for ROCm (fused MoE / GEMM / attention kernels).

It is stacked on top of the apt-installed **ROCm 7.12 SDK** and the PyTorch wheel already shipped in `auplc-base`, and targets the **AMD Radeon™ 8060S iGPU (Strix Halo, `gfx1151`, RDNA 3.5)**.


## 1. Sanity-check the environment

Confirm the GPU is visible and the Python stack is intact before launching the server.

In [ ]:
import importlib
import importlib.metadata as md
import platform

import torch

print(f"python       : {platform.python_version()}")
for m in ("torch", "triton", "aiter", "vllm", "ray"):
    mod = importlib.import_module(m)
    print(f"{m:12s} : {getattr(mod, '__version__', '?')}")
print(f"flash_attn   : {md.version('flash_attn')}  (import deferred — needs /dev/kfd)")

print()
print(f"torch.version.hip       : {torch.version.hip}")
print(f"torch.cuda.is_available : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"device 0                : {torch.cuda.get_device_name(0)}")
    print(f"arch                    : {p.gcnArchName}")
    print(f"total memory            : {p.total_memory / 2**30:.1f} GiB")

In [ ]:
!rocminfo | grep -E 'Name:|gfx[0-9]' | head -n 20
!rocm-smi --showproductname --showmeminfo vram 2>/dev/null || true

## 2. Start the vLLM OpenAI-compatible server

`./server.sh` is a thin wrapper around `python -m vllm.entrypoints.openai.api_server`. Defaults:

| Env | Default | Notes |
|---|---|---|
| `MODEL` | `Qwen/Qwen3-4B` | Any HF repo or local path. |
| `DTYPE` | `bfloat16` | |
| `MAX_MODEL_LEN` | `2048` | |
| `GPU_MEM_UTIL` | `0.90` | Fraction of GPU memory the KV-cache may consume. |
| `PORT` | `8000` | |
| `EXTRA_ARGS` | `--trust-remote-code --no-enable-log-requests` | Passed straight through. |

For this walkthrough we'll use a smaller model (`Qwen/Qwen3-1.7B`) so it warms up fast on Strix Halo. Override `MODEL` to switch.

In [ ]:
import os
import pathlib
import signal
import subprocess

MODEL = os.environ.get("MODEL", "Qwen/Qwen3-1.7B")
PORT = "8000"
LOG = pathlib.Path("server.log")

server_env = {
    **os.environ,
    "MODEL": MODEL,
    "PORT": PORT,
    "MAX_MODEL_LEN": "2048",
    "GPU_MEM_UTIL": "0.90",
}

server = subprocess.Popen(
    ["bash", "./server.sh"],
    stdout=LOG.open("w"),
    stderr=subprocess.STDOUT,
    env=server_env,
    preexec_fn=os.setsid,
)
print(f"server pid={server.pid}  model={MODEL}  port={PORT}")
print(f"log -> {LOG.resolve()} (tail it in a terminal to watch warmup)")

In [ ]:
import time

import requests

BASE_URL = f"http://127.0.0.1:{PORT}"
deadline = time.time() + 600
last_err = None
while time.time() < deadline:
    if server.poll() is not None:
        raise RuntimeError(
            f"server exited early (returncode={server.returncode}); see {LOG}"
        )
    try:
        r = requests.get(f"{BASE_URL}/v1/models", timeout=2)
        if r.ok:
            print("ready:", r.json())
            break
    except requests.RequestException as exc:
        last_err = exc
    time.sleep(3)
else:
    raise RuntimeError(f"server did not come up in 600s — last error: {last_err}; see {LOG}")

## 3. Send a chat completion

Standard OpenAI Chat Completions schema — works with the `openai` Python SDK, `curl`, LangChain, etc. Below we use plain `requests` to keep the dependency graph small.

In [ ]:
resp = requests.post(
    f"{BASE_URL}/v1/chat/completions",
    json={
        "model": MODEL,
        "messages": [
            {"role": "system", "content": "You are a concise technical assistant."},
            {"role": "user", "content": "In one paragraph, what is AMD Strix Halo and why is its iGPU interesting for LLM inference?"},
        ],
        "max_tokens": 200,
        "temperature": 0.2,
    },
    timeout=180,
)
resp.raise_for_status()
out = resp.json()
print(out["choices"][0]["message"]["content"])
print()
print("usage:", out["usage"])

In [ ]:
import json

resp = requests.post(
    f"{BASE_URL}/v1/chat/completions",
    json={
        "model": MODEL,
        "messages": [{"role": "user", "content": "Count from 1 to 5."}],
        "max_tokens": 60,
        "stream": True,
    },
    stream=True,
    timeout=120,
)
resp.raise_for_status()
for raw in resp.iter_lines():
    if not raw or not raw.startswith(b"data: "):
        continue
    payload = raw[len(b"data: ") :]
    if payload == b"[DONE]":
        print()
        break
    chunk = json.loads(payload)
    delta = chunk["choices"][0].get("delta", {}).get("content", "")
    print(delta, end="", flush=True)

## 4. Benchmark — `vllm bench serve`

`./bench.sh` wraps `vllm bench serve` against `${BASE_URL}` (mode `serve`) or `vllm bench throughput` for offline runs (mode `throughput`).

Defaults are tuned for the full SLA report (500 prompts × 1024-in / 512-out, unbounded concurrency) — that takes several minutes on Strix Halo. For a quick sanity run we shrink the workload below. Results land as JSON in `${RESULT_DIR}` (default `${HOME}/results`).

In [ ]:
import os
import pathlib
import subprocess

RESULT_DIR = pathlib.Path(os.environ.get("HOME", "/tmp")) / "results"
RESULT_DIR.mkdir(parents=True, exist_ok=True)

subprocess.run(
    ["bash", "./bench.sh"],
    check=True,
    env={
        **os.environ,
        "MODE": "serve",
        "MODEL": MODEL,
        "BASE_URL": BASE_URL,
        "NUM_PROMPTS": "16",
        "INPUT_LEN": "256",
        "OUTPUT_LEN": "128",
        "RESULT_DIR": str(RESULT_DIR),
        "WAIT_FOR_SERVER": "1",
        "MAX_WAIT": "30",
    },
)

print()
print("results in:", RESULT_DIR)
for p in sorted(RESULT_DIR.glob("*.json"))[-3:]:
    print(" -", p.name)

## 5. Stop the server

Always terminate the process group — `server.sh` `exec`s `python -m vllm.entrypoints.openai.api_server`, which itself spawns a worker process that won't go away with a plain `SIGTERM` to the parent shell.

In [ ]:
try:
    os.killpg(os.getpgid(server.pid), signal.SIGTERM)
    server.wait(timeout=30)
except ProcessLookupError:
    pass
except subprocess.TimeoutExpired:
    os.killpg(os.getpgid(server.pid), signal.SIGKILL)
    server.wait(timeout=10)

print(f"server stopped (returncode={server.returncode})")